In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

DATA_PATH    = "synthetic_wsi_delhi_engineered.csv"
TARGET       = "safety_index"
TEST_SIZE    = 0.20
RANDOM_STATE = 42
NUM_ROUNDS   = 400
EARLY_STOP   = 25
MODEL_OUT    = "xgboost_wsi_model.json"
PRED_OUT     = "predictions_wsi.csv"
FIMP_OUT     = "feature_importances_wsi.csv"


def rmse(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))


# ── Load data ──
df = pd.read_csv(DATA_PATH)

# Prepare numeric features only (label-encoded categoricals are already numeric)
X_all = df[[c for c in df.columns if c != TARGET]].select_dtypes(include=[np.number]).fillna(0.0)
y_all = df[TARGET].values
feature_names = X_all.columns.tolist()
print("Using numeric features:", len(feature_names))
print("Features:", feature_names)

# ── Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print("Train/test shapes:", X_train.shape, X_test.shape)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval   = xgb.DMatrix(X_test, label=y_test, feature_names=feature_names)

params = {
    "objective": "reg:squarederror",
    "eta": 0.05,
    "max_depth": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "seed": RANDOM_STATE,
    "verbosity": 1
}

# ── Train ──
t0 = time.time()
model = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=[(dtrain, "train"), (dval, "eval")],
    early_stopping_rounds=EARLY_STOP,
    verbose_eval=50
)
t_elapsed = time.time() - t0
print(f"Training done in {t_elapsed:.1f}s. Best iter: {getattr(model, 'best_iteration', None)}")

# ── Predict ──
if getattr(model, "best_iteration", None) is not None:
    y_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
else:
    y_pred = model.predict(dval)

# ── Metrics ──
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rm = rmse(y_test, y_pred)
print("\nEval:")
print(f"R2:  {r2:.6f}")
print(f"MAE: {mae:.6f}")
print(f"RMSE:{rm:.6f}")

# ── Save model ──
model.save_model(MODEL_OUT)
print(f"\nModel saved to {MODEL_OUT}")

pd.DataFrame({"y_true": y_test, "y_pred": y_pred}).to_csv(PRED_OUT, index=False)
print(f"Predictions saved to {PRED_OUT}")

# ── Feature importances ──
def simple_importances(xgb_model, feat_names):
    booster = xgb_model if isinstance(xgb_model, xgb.core.Booster) else xgb_model.get_booster()
    try:
        gain = booster.get_score(importance_type="gain") or {}
    except Exception:
        gain = {}
    try:
        weight = booster.get_score(importance_type="weight") or {}
    except Exception:
        weight = {}
    keys = sorted(set(list(gain.keys()) + list(weight.keys())))
    rows = []
    for k in keys:
        rows.append({
            "feature": k,
            "gain": float(gain.get(k, 0.0)),
            "weight": float(weight.get(k, 0.0))
        })
    df_imp = pd.DataFrame(rows)
    if not df_imp.empty and df_imp["feature"].str.match(r"f\d+$").all() and feat_names is not None:
        def map_fn(fn):
            idx = int(fn[1:])
            return feat_names[idx] if idx < len(feat_names) else fn
        df_imp["feature"] = df_imp["feature"].apply(map_fn)
    if not df_imp.empty:
        df_imp = df_imp.sort_values("gain", ascending=False).reset_index(drop=True)
    return df_imp

imp_df = simple_importances(model, feature_names)
if imp_df.empty and hasattr(model, "feature_importances_"):
    vals = model.feature_importances_
    if len(vals) == len(feature_names):
        imp_df = pd.DataFrame({"feature": feature_names, "gain": vals, "weight": vals}).sort_values("gain", ascending=False)

imp_df.to_csv(FIMP_OUT, index=False)
print(f"Feature importances saved to {FIMP_OUT}")
print("\nTop 15 features by gain:")
print(imp_df.head(15).to_string(index=False))

# ── Check if lux features are important ──
new_features = {"lux_reading", "expected_night_lux", "lux_anomaly", "shadow_penalty",
                "lux_x_night", "lux_severity", "location_type_enc", "is_daytime"}
top15 = set(imp_df.head(15)["feature"].tolist())
lux_in_top = new_features & top15
if lux_in_top:
    print(f"\n[OK] New features in top 15: {lux_in_top}")
else:
    print(f"\n[INFO] No new lux features in top 15 yet — they may have moderate but useful importance")

# Show all new feature ranks
print("\nNew feature rankings:")
for feat in sorted(new_features):
    rank_row = imp_df[imp_df["feature"] == feat]
    if not rank_row.empty:
        rank = rank_row.index[0] + 1
        gain = rank_row.iloc[0]["gain"]
        print(f"  {feat}: rank #{rank}, gain={gain:.4f}")
    else:
        print(f"  {feat}: not found in model features")
